# v82 -- IceCube IC86-II Hemispheric Asymmetry (REAL DATA)
## KTF³-R axis test: l=277°, b=-31°
**Author:** Andrew Cotting | 5 April 2026 | github.com/Andrew-Cot/KTF3-Analysis

Data: IceCube IC86-II_exp.csv | 112858 events | DOI: 10.21234/sxvs-mt83
Columns: MJD[days], log10(E/GeV), AngErr[deg], RA[deg], Dec[deg], Azimuth[deg], Zenith[deg]
Previous result v71: sigma=+2.61

In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd, os

print('='*60)
print('v82 -- IceCube IC86-II Hemispheric Asymmetry (REAL DATA)')
print('KTF³ axis: l=277°, b=-31°  |  v71 ref: +2.61')
print('='*60)

# Find IceCube file
ic_file = None
for fname in sorted(os.listdir('.')):
    if 'IC86' in fname and fname.endswith('.csv'):
        size = os.path.getsize(fname)
        print(f'Found: {fname} ({size/1e6:.1f} MB)')
        if size > 1e6:
            ic_file = fname
            break

if ic_file is None:
    raise FileNotFoundError('Upload IC86_II_exp (1).csv to Colab!')

# Load — first line is header with # prefix
df = pd.read_csv(ic_file, sep=r'\s+', comment='#', header=None,
                 names=['MJD','log10E','AngErr','RA','Dec','Azimuth','Zenith'])

# Remove header row if it snuck in as data
df = df[pd.to_numeric(df['MJD'], errors='coerce').notna()]
df = df.astype(float)
print(f'Loaded: {len(df):,} events')
print(f'RA:  {df["RA"].min():.1f} - {df["RA"].max():.1f} deg')
print(f'Dec: {df["Dec"].min():.1f} - {df["Dec"].max():.1f} deg')
print(f'log10E: {df["log10E"].min():.1f} - {df["log10E"].max():.1f}')


In [ ]:
def radec_to_gal(ra_deg, dec_deg):
    ra_r = np.radians(ra_deg); dec_r = np.radians(dec_deg)
    ra_NGP = np.radians(192.85948); dec_NGP = np.radians(27.12825)
    sin_b = (np.sin(dec_r)*np.sin(dec_NGP) +
             np.cos(dec_r)*np.cos(dec_NGP)*np.cos(ra_r - ra_NGP))
    b = np.degrees(np.arcsin(np.clip(sin_b, -1, 1)))
    x = np.cos(dec_r)*np.sin(ra_r - ra_NGP)
    y = (np.sin(dec_r)*np.cos(dec_NGP) -
         np.cos(dec_r)*np.sin(dec_NGP)*np.cos(ra_r - ra_NGP))
    l = (122.93192 - np.degrees(np.arctan2(x, y))) % 360
    return l, b

mask = np.isfinite(df['RA'].values) & np.isfinite(df['Dec'].values)
ra  = df['RA'].values[mask]
dec = df['Dec'].values[mask]
log10E = df['log10E'].values[mask]
print(f'Valid events: {mask.sum():,}')

l_gal, b_gal = radec_to_gal(ra, dec)

# KTF³ axis
l_k, b_k = np.radians(277.0), np.radians(-31.0)
axis = np.array([np.cos(b_k)*np.cos(l_k),
                 np.cos(b_k)*np.sin(l_k),
                 np.sin(b_k)])

l_r = np.radians(l_gal); b_r = np.radians(b_gal)
vecs = np.column_stack([np.cos(b_r)*np.cos(l_r),
                        np.cos(b_r)*np.sin(l_r),
                        np.sin(b_r)])
dot = vecs @ axis

# Raw sigma
nn, ns = np.sum(dot>0), np.sum(dot<0)
asym = (nn-ns)/(nn+ns)
sigma_raw = asym * np.sqrt(nn+ns)
print(f'N north: {nn:,}, N south: {ns:,}')
print(f'sigma (raw): {sigma_raw:+.3f}')

# RA scrambling
print('RA scrambling (500 iter)...')
np.random.seed(277)
scram = []
for _ in range(500):
    ra_s = np.random.uniform(0, 360, len(ra))
    ls, bs = radec_to_gal(ra_s, dec)
    lrs = np.radians(ls); brs = np.radians(bs)
    vs = np.column_stack([np.cos(brs)*np.cos(lrs),
                          np.cos(brs)*np.sin(lrs),
                          np.sin(brs)])
    d = vs @ axis
    scram.append((np.sum(d>0)-np.sum(d<0))/(np.sum(d>0)+np.sum(d<0)))
scram = np.array(scram)
sigma_corr = (asym - scram.mean()) / scram.std()
p_val = np.mean(np.abs(scram-scram.mean()) >= np.abs(asym-scram.mean()))

print(f'sigma (corrected): {sigma_corr:+.3f}')
print(f'p-value: {p_val:.3f}')
verdict = 'SIGNAL' if abs(sigma_corr)>2 else ('HINT' if abs(sigma_corr)>1 else 'NULL')
print(f'VERDICT: {verdict}  (v71 ref: +2.61)')

# Energy bins
print('\nEnergy bins:')
sigma_E = []; e_centers = []
for lo, hi in [(2,3),(3,4),(4,5),(5,6)]:
    me = (log10E>=lo)&(log10E<hi)
    if me.sum()<100: continue
    obs_e = (np.sum(dot[me]>0)-np.sum(dot[me]<0))/(np.sum(dot[me]>0)+np.sum(dot[me]<0))
    sc_e = []
    dec_e = dec[me]
    for _ in range(300):
        ra_s = np.random.uniform(0,360,len(dec_e))
        ls,bs = radec_to_gal(ra_s,dec_e)
        lrs=np.radians(ls); brs=np.radians(bs)
        vs=np.column_stack([np.cos(brs)*np.cos(lrs),
                            np.cos(brs)*np.sin(lrs),
                            np.sin(brs)])
        d=vs@axis; n1=np.sum(d>0); n2=np.sum(d<0)
        sc_e.append((n1-n2)/(n1+n2))
    sc_e=np.array(sc_e)
    s=(obs_e-sc_e.mean())/sc_e.std()
    sigma_E.append(s); e_centers.append(0.5*(lo+hi))
    print(f'  log10E={lo}-{hi} (N={me.sum():,}): sigma={s:+.2f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,6))
fig.suptitle(
    f'v82 -- IceCube IC86-II | REAL DATA | N={nn+ns:,}\n'
    f'KTF³: l=277°, b=-31° | σ(raw)={sigma_raw:+.2f} | '
    f'σ(corr)={sigma_corr:+.2f} | p={p_val:.3f} [{verdict}]\n'
    f'v71 reference: +2.61',
    fontsize=11, fontweight='bold'
)

ax1=axes[0]
ax1.hist(dot, bins=40, color='#3498db', alpha=0.7, edgecolor='black')
ax1.axvline(0, color='red', lw=2.5, linestyle='--', label='KTF³ equator')
ax1.set_xlabel('cos(angle) with KTF³ axis', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title(f'Hemispheric distribution\nσ(raw)={sigma_raw:+.3f}', fontsize=11)
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2=axes[1]
ax2.hist(scram, bins=30, color='#95a5a6', alpha=0.8, edgecolor='black',
         density=True, label='RA scrambled (null)')
ax2.axvline(asym, color='green', lw=2.5, label=f'Obs σ={sigma_corr:+.2f}')
ax2.axvline(scram.mean()+2*scram.std(), color='red', linestyle=':', label='2σ')
ax2.axvline(scram.mean()-2*scram.std(), color='red', linestyle=':')
ax2.set_xlabel('Hemispheric asymmetry', fontsize=11)
ax2.set_title(f'RA scrambling null | p={p_val:.3f} [{verdict}]', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

ax3=axes[2]
if sigma_E:
    ax3.plot(e_centers, sigma_E, 'b-o', lw=2, ms=9)
    ax3.axhline(0, color='black')
    ax3.axhline(1, color='orange', linestyle='--', label='1σ')
    ax3.axhline(-1, color='orange', linestyle='--')
    ax3.axhline(2.61, color='green', linestyle='--', lw=2, label='v71 +2.61')
ax3.set_xlabel('log₁₀(E/GeV)', fontsize=11)
ax3.set_ylabel('Sigma (corrected)', fontsize=11)
ax3.set_title('Energy-dependent asymmetry', fontsize=11)
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v82_icecube_updated.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'v82 FINAL: σ(raw)={sigma_raw:+.3f}, σ(corr)={sigma_corr:+.3f}, p={p_val:.3f}, {verdict}')
print('GitHub: github.com/Andrew-Cot/KTF3-Analysis | OSF: osf.io/42nks')
